# Appendix B — Framework Comparison

*Now that you have built every piece by hand, what does a framework give you?*

## Objective

Compare the glassloop assembly with how the same workflow would look in popular frameworks. We focus on three axes that matter for production: lines of code, observability and governance affordances.

## The glassloop assembly (recap)

Build a governed tool-calling agent in roughly 12 lines.

In [ ]:
from glassloop.core import BaseAgent, Finish, TaskSpec, ToolCall
from glassloop.governance import GovernanceHarness, PolicyEngine, pii_policy
from glassloop.tools import GovernedToolExecutor, ToolRegistry

In [ ]:
class StubAgent(BaseAgent):
    def propose_action(self, state):
        return Finish(output='handled')

registry = ToolRegistry()
executor = GovernedToolExecutor(registry, gates=[PolicyEngine([pii_policy]).as_gate()])
harness = GovernanceHarness(StubAgent(), executor)
traj = harness.run(TaskSpec(goal='hello'))
print('status:', traj.final_state.status, 'audit:', harness.audit.verify())

## What a LangChain version would look like

*Illustrative only — not executable here.* The shape of the API is what matters.

```python
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate
from langchain_anthropic import ChatAnthropic

tools = [...]
prompt = PromptTemplate.from_template("... do not include PII ...")
llm = ChatAnthropic(model='claude-opus-4-6')
agent = create_react_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
result = executor.invoke({'input': 'hello'})
```

The framework hides the agent loop, the tool selection logic and the policy gates. Policies are expressed in the prompt rather than as code. There is no native audit chain — observability comes from LangSmith or your own logging.

## Side-by-side

| Aspect | glassloop | framework (typical) |
| --- | --- | --- |
| Lines of code for the assembly | ~12 | ~6 |
| Where the agent loop lives | `core.loop` (10 lines, readable) | inside the framework |
| Where policies live | Python callables | system prompt |
| Policy enforcement | runtime gates | model interpretation |
| Audit | hash-chained event log | external (LangSmith etc.) |
| Trajectory observability | `StepRecord` list | callbacks or tracer |
| Tool schemas | required pydantic models | usually optional |
| Cost | manual + adapter | usually framework-tracked |

Frameworks are great when their abstractions match your problem. They are terrible when you have to reverse-engineer their internals to add a single gate or audit field. Now that you've built every piece, you can read the framework source and recognize each abstraction.

## Recommendation

Build with `glassloop` (or your own equivalent) when:
- You need policy-as-code that auditors can read.
- You operate in a regulated domain.
- You need trajectory-level evaluation and tamper-evident logs.

Adopt a framework when:
- You're prototyping and the policy surface is small.
- The team is already trained on the framework.
- The framework's escape hatches match the small set of places you actually need to customize.

Avoid the failure mode of either dogma. *"Just use the framework"* and *"frameworks are bad"* are both wrong.

## Anti-patterns flagged here

- Adopting a framework before you understand what it abstracts.
- Re-implementing the world for the sake of avoiding a dependency.
- Mixing two frameworks in the same agent.

In [ ]:
# Self-check: the glassloop assembly above ran and the audit chain verifies.
assert traj.final_state.status == 'done'
assert harness.audit.verify()
print('OK')